# Stage N — NLI pretraining через Multiple Negatives Ranking Loss

## Сравнение двух идей

| | Подход 1 (картинка): NLI-классификация поверх frozen encoder | Подход 2: контрастивный InfoNCE на entailment-парах |
|---|---|---|
| Что обучается | Linear(1024→1024) проекция + classifier(3072→3) | LoRA-адаптеры самого encoder'а |
| Лосс | Cross-entropy на 3 класса (entail/neutral/contradiction) | Симметричный InfoNCE: positive — entailment, negatives — in-batch + contradiction |
| Использует ли всю разметку | Да (3 класса) | Только entailment как positives, contradiction как hard negatives — neutral отбрасывается |
| Передаётся ли в Stage B/C | **Нет** — encoder заморожен, обучается только надстройка | **Да** — LoRA-веса напрямую загружаются в следующую стадию |
| Прямо оптимизирует cos similarity | Нет (CE на logits классификатора) | Да (InfoNCE = NT-Xent на cos) |
| Соответствие реальности | Это InferSent/SBERT-style | Это **именно** как обучали E5/BGE на NLI |

### Решение

Беру **Подход 2** как основной, потому что:
1. Прямо совместим с нашим каскадом — даёт LoRA-чекпойнт для Stage B/C.
2. Прямо оптимизирует то, что измеряет STS (cosine).
3. Не плодит лишних обучаемых параметров вне модели.

### Важная добавка к подходу 2 — hard negatives из `contradiction`

Чистый MNRL на entailment-парах теряет разметку `label==2`. Я её использую: для каждого `(premise, entailment_hypothesis)` мы берём `contradiction_hypothesis` того же premise — это **идеальный hard negative**: лексически близкий, семантически противоположный. Технически это превращает MNRL в InfoNCE с дополнительными explicit hard negatives, что даёт более сильный сигнал.

## Источники NLI

NLI (переводные наборы, включая SNLI/MNLI): cointegrated/nli-rus-translated-v2021 https://huggingface.co/datasets/cointegrated/nli-rus-translated-v2021
* **ru-WANLI** (`deepvk/ru-wanli`) — ~110k русскоязычных пар. Критично, чтобы модель не «забыла» русский.

Все три объединяются, обрабатываются единообразно (label-конвенция: `0=entail, 1=neutral, 2=contradict`).

## Промпты по моделям

| Модель | Pooling | Промпт |
|---|---|---|
| `deepvk/USER-bge-m3` | CLS | — |
| `intfloat/multilingual-e5-large-instruct` | mean | `Instruct: Given a text, retrieve relevant passages\nQuery: {t}` |
| `Qwen/Qwen3-Embedding-0.6B` | last-token | `Instruct: Retrieve semantically similar text\nQuery: {t}` |

## На выходе

Чекпойнты в `./checkpoints_stageN/{model}__stageN.pt` — формат `STSEncoder`-совместимый, готовы к загрузке в Stage B (с парафразами) или Stage C (с STS-данными).

In [ ]:
# !pip uninstall -y torch torchvision torchaudio torchao
# !pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu121
# !pip install -U "transformers>=4.40" "datasets>=2.18" "peft==0.12.0" "accelerate" "scipy" "tqdm"

In [ ]:
import os, math, random, json, gc as _gc
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType, get_peft_model_state_dict, set_peft_model_state_dict
from scipy.stats import pearsonr, spearmanr
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ----- Конфигурация -----
USE_DORA       = True
CKPT_DIR       = "./checkpoints_stageN"
HISTORY_DIR    = "./histories_stageN"
USE_HARD_CONTRADICT = True   # добавлять contradiction как явный hard negative
EVAL_EVERY_FRAC = 0.1       # eval на STS каждые 25% эпохи (4 раза за эпоху)

os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(HISTORY_DIR, exist_ok=True)
print(f"USE_DORA={USE_DORA}, MAX_NLI_PAIRS={MAX_NLI_PAIRS}, "
      f"USE_HARD_CONTRADICT={USE_HARD_CONTRADICT}")
print(f"Checkpoints -> {CKPT_DIR}")


Using device: cuda
GPU: NVIDIA A100-SXM4-40GB
USE_DORA=True, MAX_NLI_PAIRS=100000, USE_HARD_CONTRADICT=True
Checkpoints -> ./checkpoints_stageN


In [ ]:
# ----- Eval datasets: STSB-ru (dev/test) и STS22-ru (test).  -----
def sts_quality(y_true, y_pred):
    p = pearsonr(y_true, y_pred)[0]
    s = spearmanr(y_true, y_pred)[0]
    return float((p + s) / 2.0)

stsb  = load_dataset("PhilipMay/stsb_multi_mt", "ru")
sts22 = load_dataset("mteb/sts22-crosslingual-sts", "ru")

def _prep(ds, s1c="sentence1", s2c="sentence2", yc="similarity_score"):
    s1 = [x[s1c] for x in ds]; s2 = [x[s2c] for x in ds]
    y  = np.array([float(x[yc]) for x in ds], dtype=np.float32)
    return s1, s2, y

val_s1,  val_s2,  val_y                     = _prep(stsb["dev"])
test_s1, test_s2, test_y                    = _prep(stsb["test"])
sts22_test_s1, sts22_test_s2, sts22_test_y  = _prep(sts22["test"], "sentence1", "sentence2", "score")
print(f"STSB dev={len(val_s1)}, test={len(test_s1)}; STS22 test={len(sts22_test_s1)}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

ru/train-00000-of-00001.parquet:   0%|          | 0.00/721k [00:00<?, ?B/s]

ru/test-00000-of-00001.parquet:   0%|          | 0.00/158k [00:00<?, ?B/s]

ru/dev-00000-of-00001.parquet:   0%|          | 0.00/209k [00:00<?, ?B/s]

...

Generating dev split:   0%|          | 0/1500 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

test/ru.jsonl.gz:   0%|          | 0.00/496k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/265 [00:00<?, ? examples/s]

STSB dev=1500, test=1379; STS22 test=265


In [ ]:
from datasets import load_dataset
from typing import List, Tuple
import random

# ----- NLI data: SNLI + MultiNLI + ru-WANLI с единой схемой меток -----
# Конвенция меток везде: 0=entailment, 1=neutral, 2=contradiction
# Спорные/отсутствующие метки (-1, "-", None) выкидываем.

LABEL_MAP = {"entailment": 0, "neutral": 1, "contradiction": 2, "0": 0, "1": 1, "2": 2}

def _is_valid_label(lab):
    if lab is None or lab == "" or lab == "-" or lab == -1:
        return False
    lab_str = str(lab).lower().strip()
    return lab_str in LABEL_MAP

def _norm_label(lab):
    lab_str = str(lab).lower().strip()
    return LABEL_MAP[lab_str]

def _load_nli_source(name: str, loader_fn) -> List[Tuple[str, str, int]]:
    try:
        rows = loader_fn()
        out = []
        for ex in rows:
            if not _is_valid_label(ex["label"]): continue
            p = ex["premise"]; h = ex["hypothesis"]
            if not isinstance(p, str) or not isinstance(h, str): continue
            if len(p.strip()) < 3 or len(h.strip()) < 3: continue
            out.append((p, h, _norm_label(ex["label"])))
        print(f"  {name}: {len(out)} valid rows")
        return out
    except Exception as e:
        print(f"  {name}: failed ({type(e).__name__}: {e})")
        return []

# cointegrated/nli-rus-translated-v2021: ru_* -> premise/hypothesis
def load_nli_rus_translated(split: str = "train"):
    ds = load_dataset("cointegrated/nli-rus-translated-v2021", split=split)
    def map_columns(ex):
        ex["premise"] = ex["premise_ru"]
        ex["hypothesis"] = ex["hypothesis_ru"]
        return ex
    return ds.map(map_columns)

# ru-WANLI: стандарт
def load_ru_wanli(split: str = "train"):
    return load_dataset("deepvk/ru-WANLI", split=split)

print("Loading NLI sources...")
all_rows: List[Tuple[str, str, int]] = []

all_rows += _load_nli_source(
    "ru-WANLI/train",
    lambda: load_ru_wanli("train"),
)

all_rows += _load_nli_source(
    "NLI/train (ru-translated)",
    lambda: load_nli_rus_translated("train"),
)

random.shuffle(all_rows)
print(f"\nTotal NLI rows (combined, shuffled): {len(all_rows)}")
print(f"  label distribution: "
      f"entail={sum(1 for r in all_rows if r[2]==0)}, "
      f"neutral={sum(1 for r in all_rows if r[2]==1)}, "
      f"contradict={sum(1 for r in all_rows if r[2]==2)}")


Loading NLI sources...


README.md: 0.00B [00:00, ?B/s]

data/train.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

data/val.parquet:   0%|          | 0.00/391k [00:00<?, ?B/s]

data/test.parquet:   0%|          | 0.00/791k [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/1756548 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/106557 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/34615 [00:00<?, ? examples/s]

Map:   0%|          | 0/1756548 [00:00<?, ? examples/s]

  NLI/train (ru-translated): 1511454 valid rows

Total NLI rows (combined, shuffled): 1611454
  label distribution: entail=702581, neutral=465185, contradict=443688


In [ ]:
# ----- Извлекаем entailment-пары + словарь premise -> contradictions для hard negs -----
# Для каждого premise собираем его entailments (positives) и contradictions (hard negs).
# Если для якоря-premise есть contradiction-гипотеза — используем её как явный hard negative.

prem_to_entail: Dict[str, List[str]]   = defaultdict(list)
prem_to_contra: Dict[str, List[str]]   = defaultdict(list)

for (p, h, lab) in all_rows:
    if lab == 0:
        prem_to_entail[p].append(h)
    elif lab == 2:
        prem_to_contra[p].append(h)

# Собираем плоский список entailment-пар
entail_pairs: List[Tuple[str, str, Optional[str]]] = []
for p, hyps in prem_to_entail.items():
    contras = prem_to_contra.get(p, [])
    for h in hyps:
        # Если есть contradiction для этого же premise — берём один как hard negative
        cn = random.choice(contras) if contras else None
        entail_pairs.append((p, h, cn))

random.shuffle(entail_pairs)
if MAX_NLI_PAIRS is not None and len(entail_pairs) > MAX_NLI_PAIRS:
    entail_pairs = entail_pairs[:MAX_NLI_PAIRS]

n_with_hn = sum(1 for _, _, cn in entail_pairs if cn is not None)
print(f"Entailment pairs (positives): {len(entail_pairs)}")
print(f"  with contradiction-hard-negative: {n_with_hn} ({100*n_with_hn/max(1,len(entail_pairs)):.1f}%)")
print(f"  without hard-negative: {len(entail_pairs) - n_with_hn}")

# В удобный формат
nli_a    = [p for p, h, cn in entail_pairs]
nli_b    = [h for p, h, cn in entail_pairs]
nli_hn   = [cn for p, h, cn in entail_pairs]

# Освобождаем память
del all_rows, prem_to_entail, prem_to_contra; _gc.collect()


Entailment pairs (positives): 100000
  with contradiction-hard-negative: 62589 (62.6%)
  without hard-negative: 37411


0

In [ ]:
# ----- Gradient Cache c поддержкой явных hard negatives -----
class GradientCache:
    def __init__(self, model, optimizer, chunk_size=16, temperature=0.05):
        self.model = model
        self.optimizer = optimizer
        self.chunk_size = chunk_size
        self.temperature = temperature

    @torch.no_grad()
    def _encode_no_grad(self, texts):
        parts = []
        for i in range(0, len(texts), self.chunk_size):
            parts.append(self.model(texts[i:i+self.chunk_size]).detach())
        return torch.cat(parts, 0)

    def step(self, texts_a, texts_b, hard_neg_texts=None):
        was = self.model.training
        self.model.eval()
        q_full = self._encode_no_grad(texts_a)
        p_full = self._encode_no_grad(texts_b)
        n_full = self._encode_no_grad(hard_neg_texts) if hard_neg_texts else None
        self.model.train(was)

        q = q_full.detach().requires_grad_(True)
        p = p_full.detach().requires_grad_(True)
        if n_full is not None:
            n = n_full.detach().requires_grad_(True)
            bank = torch.cat([p, n], 0)
        else:
            n = None; bank = p

        logits = (q @ bank.T) / self.temperature
        labels = torch.arange(q.size(0), device=q.device)
        loss_fwd = F.cross_entropy(logits, labels)
        loss_bwd = F.cross_entropy((p @ q.T) / self.temperature, labels)
        loss = 0.5 * (loss_fwd + loss_bwd)
        loss.backward()
        q_grad = q.grad.detach(); p_grad = p.grad.detach()
        n_grad = n.grad.detach() if n is not None else None
        loss_value = loss.item()
        del q, p, n, bank, logits, loss

        self.optimizer.zero_grad(set_to_none=True)
        for i in range(0, len(texts_a), self.chunk_size):
            qc = self.model(texts_a[i:i+self.chunk_size])
            pc = self.model(texts_b[i:i+self.chunk_size])
            sg = (qc * q_grad[i:i+self.chunk_size]).sum() + (pc * p_grad[i:i+self.chunk_size]).sum()
            sg.backward()
            del qc, pc, sg
        if hard_neg_texts:
            for i in range(0, len(hard_neg_texts), self.chunk_size):
                nc = self.model(hard_neg_texts[i:i+self.chunk_size])
                sg = (nc * n_grad[i:i+self.chunk_size]).sum()
                sg.backward()
                del nc, sg

        self.optimizer.step()
        self.optimizer.zero_grad(set_to_none=True)
        return loss_value


In [ ]:
# ----- STSEncoder: поддержка двух разных Instruct-промптов (для mE5 и Qwen) -----
def _detect_lora_targets(model) -> List[str]:
    names = {n for n, _ in model.named_modules()}
    if any(".q_proj" in n for n in names):
        return ["q_proj", "k_proj", "v_proj", "o_proj"]
    return ["query", "key", "value", "output.dense"]

def last_token_pool(h, attention_mask):
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return h[:, -1]
    seq_lens = attention_mask.sum(dim=1) - 1
    bs = h.size(0)
    return h[torch.arange(bs, device=h.device), seq_lens]

# Разные task-описания для разных моделей (как ты указал)
E5_RETRIEVAL_TASK = "Given a text, retrieve relevant passages"
QWEN_STS_TASK     = "Retrieve semantically similar text"

@dataclass
class ModelSpec:
    name: str
    model_id: str
    pooling: str          # "cls" | "mean" | "last"
    prompt_mode: str      # "none" | "e5_retrieval" | "qwen_sts"
    is_qwen: bool = False
    dtype: torch.dtype = torch.float32
    max_length: int = 192
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.1


class STSEncoder(nn.Module):
    def __init__(self, spec: ModelSpec, mode: str = "lora", use_dora: bool = True):
        super().__init__()
        self.spec = spec
        self.tokenizer = AutoTokenizer.from_pretrained(
            spec.model_id, use_fast=True,
            padding_side="left" if spec.is_qwen else "right",
            trust_remote_code=True,
        )
        kw = {"trust_remote_code": True}
        if spec.dtype != torch.float32:
            kw["torch_dtype"] = spec.dtype
        base = AutoModel.from_pretrained(spec.model_id, **kw)

        if mode == "lora":
            targets = _detect_lora_targets(base)
            cfg_kwargs = dict(r=spec.lora_r, lora_alpha=spec.lora_alpha,
                              lora_dropout=spec.lora_dropout, bias="none",
                              target_modules=targets,
                              task_type=TaskType.FEATURE_EXTRACTION)
            if use_dora:
                try:
                    cfg = LoraConfig(**cfg_kwargs, use_dora=True)
                except TypeError:
                    cfg = LoraConfig(**cfg_kwargs)
                    print(f"  [warn] DoRA not supported, falling back to LoRA")
            else:
                cfg = LoraConfig(**cfg_kwargs)
            self.model = get_peft_model(base, cfg)
        elif mode == "frozen":
            self.model = base
            for p in self.model.parameters(): p.requires_grad = False
        else:
            self.model = base

    def trainable_params(self):
        n_tr  = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        n_tot = sum(p.numel() for p in self.model.parameters())
        return n_tr, n_tot

    def _prep(self, texts):
        if self.spec.prompt_mode == "e5_retrieval":
            return [f"Instruct: {E5_RETRIEVAL_TASK}\nQuery: {t}" for t in texts]
        if self.spec.prompt_mode == "qwen_sts":
            return [f"Instruct: {QWEN_STS_TASK}\nQuery: {t}" for t in texts]
        return list(texts)

    def forward(self, texts):
        texts = self._prep(texts)
        tok = self.tokenizer(texts, padding=True, truncation=True,
                             max_length=self.spec.max_length, return_tensors="pt").to(device)
        out = self.model(**tok)
        h = out.last_hidden_state
        if self.spec.pooling == "cls":
            emb = h[:, 0]
        elif self.spec.pooling == "last":
            emb = last_token_pool(h, tok["attention_mask"])
        else:
            attn = tok["attention_mask"].unsqueeze(-1).float()
            emb  = (h * attn).sum(1) / attn.sum(1).clamp(min=1e-6)
        return F.normalize(emb.float(), p=2, dim=-1)

    @torch.no_grad()
    def encode(self, texts, batch_size: int = 64, to_cpu: bool = True):
        was = self.training; self.eval()
        embs = []
        for i in range(0, len(texts), batch_size):
            e = self.forward(texts[i:i+batch_size]).detach()
            embs.append(e.cpu() if to_cpu else e)
        self.train(was)
        return torch.cat(embs, 0)

def snap(model: STSEncoder):
    sd = get_peft_model_state_dict(model.model)
    return {k: v.detach().clone().cpu() for k, v in sd.items()}

def restore(model: STSEncoder, snapshot):
    set_peft_model_state_dict(model.model, {k: v.to(device) for k, v in snapshot.items()})


In [ ]:
# ----- Eval helpers + EarlyStopper -----
@torch.no_grad()
def eval_sts(model, s1, s2, y, batch_size=64):
    e1 = model.encode(s1, batch_size=batch_size)
    e2 = model.encode(s2, batch_size=batch_size)
    return sts_quality(y, (e1 * e2).sum(-1).numpy())

def full_eval(model, batch_size=64):
    return {
        "val_stsb":   eval_sts(model, val_s1,  val_s2,  val_y, batch_size),
        "test_stsb":  eval_sts(model, test_s1, test_s2, test_y, batch_size),
        "test_sts22": eval_sts(model, sts22_test_s1, sts22_test_s2, sts22_test_y, batch_size),
    }

class EarlyStopper:
    def __init__(self, patience=2, min_delta=1e-4):
        self.patience = patience; self.min_delta = min_delta
        self.best = -1.0; self.best_step = -1; self.best_snap = None; self.cnt = 0
    def update(self, model, step, val):
        if val > self.best + self.min_delta:
            self.best = val; self.best_step = step
            self.best_snap = snap(model); self.cnt = 0
            return False
        self.cnt += 1
        return self.cnt >= self.patience


In [ ]:
# ----- Stage N: контрастивный NLI (MNRL + contradiction hard negs) -----
def train_stage_n(model: STSEncoder, anchors_a, anchors_b, hard_negs_or_none, *,
                  max_epochs: int = 2,
                  batch_size: int = 256,
                  chunk_size: int = 16,
                  lr: float = 2e-5,
                  weight_decay: float = 1e-2,
                  temperature: float = 0.05,
                  warmup_ratio: float = 0.05,
                  patience_evals: int = 3,
                  eval_every_frac: float = 0.25,
                  use_hard: bool = True,
                  eval_bs: int = 64,
                  tag: str = "run"):
    n = len(anchors_a)
    params = [p for p in model.parameters() if p.requires_grad]
    optim  = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    steps_per_epoch = max(1, n // batch_size)
    total_steps     = steps_per_epoch * max_epochs
    sched = get_cosine_schedule_with_warmup(optim, int(total_steps * warmup_ratio), total_steps)
    cache = GradientCache(model, optim, chunk_size=chunk_size, temperature=temperature)
    stopper = EarlyStopper(patience=patience_evals)

    eval_every_steps = max(1, int(steps_per_epoch * eval_every_frac))
    print(f"[{tag}] {n} pairs, {steps_per_epoch} steps/epoch, total {total_steps} steps, "
          f"eval every {eval_every_steps} steps")

    history = []
    base = full_eval(model, batch_size=eval_bs)
    base.update(step=0, epoch_frac=0.0, loss=float("nan"))
    history.append(base)
    print(f"[{tag}] baseline: val={base['val_stsb']:.4f} "
          f"test_stsb={base['test_stsb']:.4f} test_sts22={base['test_sts22']:.4f}")

    global_step = 0
    losses_window = []
    early_stop_triggered = False

    for ep in range(1, max_epochs + 1):
        model.train()
        idx = np.random.permutation(n)
        pbar = tqdm(range(0, n, batch_size),
                    desc=f"[{tag}] StageN ep {ep}/{max_epochs}", leave=False)
        for i in pbar:
            sub = idx[i:i+batch_size]
            if len(sub) < 2: continue
            a_batch = [anchors_a[j] for j in sub]
            b_batch = [anchors_b[j] for j in sub]

            hard_texts = None
            if use_hard and hard_negs_or_none is not None:
                hard = [hard_negs_or_none[j] for j in sub if hard_negs_or_none[j] is not None]
                if hard:
                    hard_texts = hard

            loss = cache.step(a_batch, b_batch, hard_neg_texts=hard_texts)
            sched.step()
            losses_window.append(loss)
            global_step += 1
            pbar.set_postfix(loss=f"{np.mean(losses_window[-50:]):.4f}")

            # --- Промежуточный eval на STS ---
            if global_step % eval_every_steps == 0:
                m = full_eval(model, batch_size=eval_bs)
                m.update(step=global_step,
                         epoch_frac=global_step / steps_per_epoch,
                         loss=float(np.mean(losses_window[-eval_every_steps:])))
                history.append(m)
                print(f"[{tag}] step {global_step} (epoch {m['epoch_frac']:.2f}): "
                      f"loss={m['loss']:.4f} val={m['val_stsb']:.4f} "
                      f"test_stsb={m['test_stsb']:.4f} test_sts22={m['test_sts22']:.4f}")

                if stopper.update(model, global_step, m["val_stsb"]):
                    print(f"[{tag}] early stop at step {global_step} "
                          f"(best val={stopper.best:.4f} @ step {stopper.best_step})")
                    early_stop_triggered = True
                    break
        if early_stop_triggered:
            break

    # Финальный eval (на случай если последний eval был не на конце эпохи)
    if not early_stop_triggered:
        m = full_eval(model, batch_size=eval_bs)
        m.update(step=global_step,
                 epoch_frac=global_step / steps_per_epoch if steps_per_epoch > 0 else 0,
                 loss=float(np.mean(losses_window[-eval_every_steps:])) if losses_window else float("nan"))
        history.append(m)
        print(f"[{tag}] final step {global_step}: val={m['val_stsb']:.4f} "
              f"test_stsb={m['test_stsb']:.4f} test_sts22={m['test_sts22']:.4f}")
        stopper.update(model, global_step, m["val_stsb"])

    if stopper.best_snap is not None:
        restore(model, stopper.best_snap)
        print(f"[{tag}] restored best-val (val={stopper.best:.4f} @ step {stopper.best_step})")
    return history, stopper


In [ ]:
# ----- Конфиги моделей и параметров обучения -----
MODEL_SPECS = {
    "USER-bge-m3": ModelSpec(
        name="USER-bge-m3", model_id="deepvk/USER-bge-m3",
        pooling="cls", prompt_mode="none",
        is_qwen=False, dtype=torch.float32, max_length=192,
        lora_r=16, lora_alpha=32, lora_dropout=0.1,
    ),
    "mE5-instruct": ModelSpec(
        name="mE5-instruct", model_id="intfloat/multilingual-e5-large-instruct",
        pooling="mean", prompt_mode="e5_retrieval",          # "Given a text, retrieve relevant passages"
        is_qwen=False, dtype=torch.float32, max_length=192,
        lora_r=16, lora_alpha=32, lora_dropout=0.1,
    ),
    "Qwen3-Emb-8B": ModelSpec(
        name="Qwen3-Emb-4B", model_id="Qwen/Qwen3-Embedding-4B",
        pooling="last", prompt_mode="qwen_sts",              # "Retrieve semantically similar text"
        is_qwen=True, dtype=torch.bfloat16, max_length=256,
        lora_r=16, lora_alpha=32, lora_dropout=0.05,
    ),
}

# По модели: батчи, чанки, LR. Виртуальный батч 256+ для XLM-R, 96 для Qwen.
TRAIN_KW = {
    "USER-bge-m3":   dict(max_epochs=1, batch_size=256, chunk_size=16, lr=2e-5,
                          temperature=0.05, patience_evals=105, eval_bs=64),
    "mE5-instruct":  dict(max_epochs=1, batch_size=192, chunk_size=12, lr=2e-5,
                          temperature=0.05, patience_evals=105, eval_bs=64),
    "Qwen3-Emb-8B":dict(max_epochs=1, batch_size=96,  chunk_size=4,  lr=1e-4,
                          temperature=0.05, patience_evals=105, eval_bs=24),
}

def free_gpu():
    _gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.synchronize()


In [ ]:
# ----- Baselines: качество frozen-моделей до Stage N -----
BASELINES = {}
for name, spec in MODEL_SPECS.items():
    print(f"\n=== Baseline: {name} ===")
    try:
        m = STSEncoder(spec, mode="frozen", use_dora=False).to(device)
        eval_bs = TRAIN_KW[name]["eval_bs"]
        BASELINES[name] = full_eval(m, batch_size=eval_bs)
        print(f"  val={BASELINES[name]['val_stsb']:.4f}  "
              f"test_stsb={BASELINES[name]['test_stsb']:.4f}  "
              f"test_sts22={BASELINES[name]['test_sts22']:.4f}")
        del m; free_gpu()
    except Exception as e:
        print(f"  !! {name} failed: {type(e).__name__}: {e}")
        BASELINES[name] = {"val_stsb": float("nan"), "test_stsb": float("nan"), "test_sts22": float("nan")}
        free_gpu()

pd.DataFrame(BASELINES).T



=== Baseline: USER-bge-m3 ===


config.json:   0%|          | 0.00/697 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  val=0.8655  test_stsb=0.8512  test_sts22=0.6800


,val_stsb,test_stsb,test_sts22
USER-bge-m3,0.857217,0.828224,0.676297
mE5-instruct,0.872453,0.843576,0.661550
Qwen3-Emb-8B,0.865456,0.851154,0.680027


In [ ]:
# ----- Главный прогон: три модели обучаются на NLI с MNRL + hard negatives -----
RESULTS: Dict[str, Dict] = {}
HISTORIES: Dict[str, List[Dict]] = {}
CKPT_PATHS: Dict[str, str] = {}

for model_name, spec in MODEL_SPECS.items():
    print(f"\n{'#'*70}\n# Stage N :: {model_name}\n{'#'*70}")
    try:
        m = STSEncoder(spec, mode="lora", use_dora=USE_DORA).to(device)
        n_tr, n_tot = m.trainable_params()
        print(f"  LoRA{'(+DoRA)' if USE_DORA else ''}: {n_tr/1e6:.2f}M / {n_tot/1e6:.1f}M trainable")

        kw = dict(TRAIN_KW[model_name])
        history, stopper = train_stage_n(
            m, nli_a, nli_b, nli_hn,
            tag=model_name,
            use_hard=USE_HARD_CONTRADICT,
            eval_every_frac=EVAL_EVERY_FRAC,
            **kw,
        )
        HISTORIES[model_name] = history

        end_metrics = full_eval(m, batch_size=kw["eval_bs"])
        base = BASELINES[model_name]
        RESULTS[model_name] = {
            "baseline":   base,
            "end":        end_metrics,
            "delta_val":  end_metrics["val_stsb"]   - base["val_stsb"],
            "delta_test": end_metrics["test_stsb"]  - base["test_stsb"],
            "delta_sts22":end_metrics["test_sts22"] - base["test_sts22"],
            "best_step":  stopper.best_step,
            "best_val":   stopper.best,
        }

        # Сохраняем чекпойнт — точка входа для последующих стадий
        save_path = os.path.join(CKPT_DIR, f"{model_name}__stageN.pt")
        torch.save({
            "state_dict": snap(m),
            "meta": {
                "spec": spec.__dict__,
                "stage": "N",
                "use_dora": USE_DORA,
                "train_kw": kw,
                "use_hard_contradict": USE_HARD_CONTRADICT,
                "n_pairs": len(nli_a),
                "baseline_metrics":  base,
                "best_metrics":      end_metrics,
                "best_step":         stopper.best_step,
                "best_val":          stopper.best,
            },
        }, save_path)
        CKPT_PATHS[model_name] = save_path
        print(f"\n  [ckpt] saved -> {save_path}")
        print(f"     Δval={RESULTS[model_name]['delta_val']:+.3f}, "
              f"Δtest={RESULTS[model_name]['delta_test']:+.3f}, "
              f"Δsts22={RESULTS[model_name]['delta_sts22']:+.3f}")

        with open(os.path.join(HISTORY_DIR, f"{model_name}__stageN.json"), "w", encoding="utf-8") as f:
            json.dump(history, f, ensure_ascii=False, indent=2, default=float)

        del m; free_gpu()
    except Exception as e:
        print(f"  !! FAILED: {type(e).__name__}: {e}")
        import traceback; traceback.print_exc()
        free_gpu()



######################################################################
# Stage N :: USER-bge-m3
######################################################################


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  LoRA(+DoRA): 5.23M / 364.3M trainable
[USER-bge-m3] 100000 pairs, 390 steps/epoch, total 390 steps, eval every 39 steps
[USER-bge-m3] baseline: val=0.8572 test_stsb=0.8282 test_sts22=0.6763


[USER-bge-m3] StageN ep 1/1:   0%|          | 0/391 [00:00<?, ?it/s]

[USER-bge-m3] step 39 (epoch 0.10): loss=0.6915 val=0.8711 test_stsb=0.8402 test_sts22=0.6771
[USER-bge-m3] step 78 (epoch 0.20): loss=0.5214 val=0.8751 test_stsb=0.8432 test_sts22=0.6757
[USER-bge-m3] step 117 (epoch 0.30): loss=0.4803 val=0.8742 test_stsb=0.8431 test_sts22=0.6733
[USER-bge-m3] step 156 (epoch 0.40): loss=0.4624 val=0.8736 test_stsb=0.8434 test_sts22=0.6720
[USER-bge-m3] step 195 (epoch 0.50): loss=0.4527 val=0.8726 test_stsb=0.8430 test_sts22=0.6708
[USER-bge-m3] step 234 (epoch 0.60): loss=0.4139 val=0.8723 test_stsb=0.8427 test_sts22=0.6699
[USER-bge-m3] step 273 (epoch 0.70): loss=0.4412 val=0.8719 test_stsb=0.8427 test_sts22=0.6684
[USER-bge-m3] step 312 (epoch 0.80): loss=0.4444 val=0.8717 test_stsb=0.8428 test_sts22=0.6682
[USER-bge-m3] step 351 (epoch 0.90): loss=0.4307 val=0.8716 test_stsb=0.8428 test_sts22=0.6681
[USER-bge-m3] step 390 (epoch 1.00): loss=0.4213 val=0.8716 test_stsb=0.8428 test_sts22=0.6681
[USER-bge-m3] final step 391: val=0.8716 test_stsb=0

...

[mE5-instruct] step 52 (epoch 0.10): loss=2.1215 val=0.8640 test_stsb=0.8280 test_sts22=0.6745
[mE5-instruct] step 104 (epoch 0.20): loss=0.7605 val=0.8572 test_stsb=0.8212 test_sts22=0.6764
[mE5-instruct] step 156 (epoch 0.30): loss=0.5106 val=0.8615 test_stsb=0.8305 test_sts22=0.6728
[mE5-instruct] step 208 (epoch 0.40): loss=0.4890 val=0.8641 test_stsb=0.8334 test_sts22=0.6649
[mE5-instruct] step 260 (epoch 0.50): loss=0.4569 val=0.8640 test_stsb=0.8344 test_sts22=0.6566
[mE5-instruct] step 312 (epoch 0.60): loss=0.4508 val=0.8635 test_stsb=0.8366 test_sts22=0.6533
[mE5-instruct] step 364 (epoch 0.70): loss=0.4421 val=0.8629 test_stsb=0.8370 test_sts22=0.6491
[mE5-instruct] step 416 (epoch 0.80): loss=0.4299 val=0.8626 test_stsb=0.8373 test_sts22=0.6472
[mE5-instruct] step 468 (epoch 0.90): loss=0.4237 val=0.8626 test_stsb=0.8373 test_sts22=0.6467
[mE5-instruct] step 520 (epoch 1.00): loss=0.4493 val=0.8625 test_stsb=0.8374 test_sts22=0.6467
[mE5-instruct] final step 521: val=0.8625

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  LoRA(+DoRA): 12.11M / 4033.9M trainable
[Qwen3-Emb-8B] 100000 pairs, 1041 steps/epoch, total 1041 steps, eval every 104 steps
[Qwen3-Emb-8B] baseline: val=0.8653 test_stsb=0.8512 test_sts22=0.6796


[Qwen3-Emb-8B] StageN ep 1/1:   0%|          | 0/1042 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ----- Графики обучения (по step внутри одной эпохи NLI) -----
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, key, title in zip(axes, ["val_stsb", "test_stsb", "test_sts22"],
                           ["val STSB (dev)", "test STSB", "test STS22"]):
    for name, hist in HISTORIES.items():
        steps = [r["step"] for r in hist]
        vals  = [r[key]   for r in hist]
        ax.plot(steps, vals, marker='o', label=name)
        # Горизонтальная линия baseline для сравнения
        base_val = BASELINES.get(name, {}).get(key, None)
        if base_val is not None:
            ax.axhline(y=base_val, linestyle=":", alpha=0.4)
    ax.set_title(title); ax.set_xlabel("Training step"); ax.set_ylabel("Quality")
    ax.legend(); ax.grid(True)
plt.suptitle("Stage N: NLI pretraining (MNRL + contradiction hard negs)")
plt.tight_layout(); plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Применяем стили из вашего примера
plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 15,
    'axes.labelsize': 13,
    'legend.fontsize': 14
})

BEST_COLORS = {
    'USER-BGE-M3 (cls)': 'blue',
    'mE5-large-instruct': 'tab:red',
    'Qwen-3-Embedding-8B': 'darkgreen'
}

BEST_MARKERS = {
    'USER-BGE-M3 (cls)': 'o',
    'mE5-large-instruct': 's',
    'Qwen-3-Embedding-8B': '^'
}

def plot_sts_best_styled(histories_dict, metric_key, title, filename, ylabel="Качество (Pearson+Spearman)/2"):
    plt.figure(figsize=(10, 6))

    # Маппинг ключей из HISTORIES в красивые названия для легенды
    best_configs = [
        ('USER-bge-m3', 'USER-BGE-M3 (cls)'),
        ('mE5-instruct', 'mE5-large-instruct'),
        ('Qwen3-Emb-8B', 'Qwen-3-Embedding-8B')
    ]

    for model_key, display_label in best_configs:
        if model_key in histories_dict:
            history = histories_dict[model_key]
            if not history: continue

            # В текущем коде используется 'epoch_frac', заменим им 'epoch' из примера
            x_axis = [r.get('epoch_frac', r.get('step', 0)) for r in history]
            values = [r[metric_key] for r in history]

            plt.plot(
                x_axis, values,
                label=display_label,
                color=BEST_COLORS.get(display_label, 'black'),
                marker=BEST_MARKERS.get(display_label, 'o'),
                linewidth=1.5,
                markersize=8
            )

    plt.title(title, pad=15)
    plt.xlabel("Эпохи (Stage N)")
    plt.ylabel(ylabel)
    plt.grid(True, linestyle='-', alpha=0.3)

    plt.legend(loc='lower right', frameon=True, facecolor='white', edgecolor='black')
    plt.tight_layout()

    plt.savefig(filename, dpi=300)
    print(f"Saved: {filename}")
    plt.show()

# Генерируем три графика по вашему запросу
if 'HISTORIES' in globals():
    plot_sts_best_styled(HISTORIES, "val_stsb", "Качество на валидационной части STSB", "fig_stsb_val_stageN.png")
    plot_sts_best_styled(HISTORIES, "test_stsb", "Качество на тестовой части STSB", "fig_stsb_test_stageN.png")
    plot_sts_best_styled(HISTORIES, "test_sts22", "Качество на STS22", "fig_sts22_test_stageN.png")
else:
    print("Ошибка: Переменная HISTORIES не найдена. Сначала запустите обучение моделей.")

In [ ]:
# ----- Сводная таблица: baseline vs Stage N -----
rows = []
for name, r in RESULTS.items():
    rows.append({
        "model": name, "stage": "baseline (frozen)", "best_step": 0,
        "val_stsb":   r["baseline"]["val_stsb"],
        "test_stsb":  r["baseline"]["test_stsb"],
        "test_sts22": r["baseline"]["test_sts22"],
    })
    rows.append({
        "model": name, "stage": f"Stage-N (best step={r['best_step']})",
        "best_step": r["best_step"],
        "val_stsb":   r["end"]["val_stsb"],
        "test_stsb":  r["end"]["test_stsb"],
        "test_sts22": r["end"]["test_sts22"],
        "Δ_val":      r["delta_val"],
        "Δ_test":     r["delta_test"],
        "Δ_sts22":    r["delta_sts22"],
        "ckpt":       CKPT_PATHS.get(name, "—"),
    })

df = pd.DataFrame(rows)
df["test_mean"] = (df["test_stsb"] + df["test_sts22"]) / 2
df_sorted = df.sort_values(["model", "best_step"]).reset_index(drop=True)
print("=== Stage N результаты ===")
df_sorted
